# NEPSE starter: prices, indices and adjusted returns

A quick tour of the [NEPSE Nepal Stock Market Data](https://www.kaggle.com/datasets/zeuton/nepse-stock-market-data) dataset:
loading the files, the NEPSE Index since 1997, why the adjusted prices matter, and returns by sector.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# the dataset folder name can differ between Kaggle's mount styles
DATA = next(p.parent for p in Path("/kaggle/input").rglob("nepse_prices.csv"))
prices = pd.read_csv(DATA / "nepse_prices.csv", parse_dates=["date"])
companies = pd.read_csv(DATA / "companies.csv")
indices = pd.read_csv(DATA / "indices.csv", parse_dates=["date"])
print(prices.shape, prices.symbol.nunique(), "companies,", prices.date.min().date(), "to", prices.date.max().date())
prices.head()

## Companies per sector

In [ ]:
companies.sector.value_counts().plot.barh(figsize=(8, 6), title="Listed companies by sector")
plt.gca().invert_yaxis(); plt.show()

## NEPSE Index since 1997

In [ ]:
nepse = indices[indices["index"] == "NEPSE Index"].set_index("date")["close"]
ax = nepse.plot(figsize=(12, 4), title="NEPSE Index (close)")
ax.set_ylabel("points"); plt.show()

## Raw vs adjusted prices

NEPSE companies issue bonus shares often, and every issue knocks the traded price down.
`adj_close` removes those drops, so use it for returns. Here is Nabil Bank:

In [ ]:
nabil = prices[prices.symbol == "NABIL"].set_index("date")
nabil[["close", "adj_close"]].plot(figsize=(12, 4), title="NABIL: traded vs adjusted close", logy=True)
plt.show()

## One-year return by sector (adjusted)

In [ ]:
end = prices.date.max()
start = end - pd.Timedelta(days=365)
px = prices.pivot_table(index="date", columns="symbol", values="adj_close")
first = px[px.index >= start].bfill().iloc[0]
last = px.ffill().iloc[-1]
ret = (last / first - 1).rename("return").to_frame().join(companies.set_index("symbol")["sector"])
ret = ret[(ret["return"].abs() < 10)]  # drop symbols with too little trading in the window
ret.groupby("sector")["return"].median().sort_values().mul(100).plot.barh(
    figsize=(8, 6), title=f"Median 1-year return by sector, to {end.date()} (%)")
plt.show()

## NPR per US dollar (Nepal Rastra Bank)

In [ ]:
fx = pd.read_csv(DATA / "forex_wide.csv", parse_dates=["date"]).set_index("date")
fx["USD"].plot(figsize=(12, 3), title="NRB buying rate, NPR per USD"); plt.show()